In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

**Dim User**

In [0]:
df = spark.read.format("parquet").load("abfss://bronze@spotifyazurestoragesink.dfs.core.windows.net/DimUser")

display(df)

**Transformation**

In [0]:
df_user_transformed = df.withColumn("user_name", upper(col("user_name")))
windowspec = Window.partitionBy("user_id").orderBy(col("updated_at").desc())
df_user_transformed = df_user_transformed.withColumn("rank", row_number().over(windowspec))
df_user_transformed = df_user_transformed.filter(col("rank") == 1)
df_user_transformed = df_user_transformed.drop("rank")

In [0]:
display(df_user_transformed)

In [0]:
df_user_transformed.write.mode("append").format("delta")\
                        .option("path", "abfss://silver@spotifyazurestoragesink.dfs.core.windows.net/DimUser/data")\
                        .saveAsTable("spotify.silver.DimUser")



**Dim Artist**

In [0]:
df_artist = spark.read.format("parquet").load("abfss://bronze@spotifyazurestoragesink.dfs.core.windows.net/DimArtist")
display(df_artist)

In [0]:
df_art_transformed = df_artist.withColumn("artist_name", upper(col("artist_name")))
windowspec = Window.partitionBy("artist_id").orderBy(col("updated_at").desc())
df_art_transformed = df_art_transformed.withColumn("rank", row_number().over(windowspec))
df_art_transformed = df_art_transformed.filter(col("rank") == 1)
df_art_transformed = df_art_transformed.drop("rank")

In [0]:
df_art_transformed.write.mode("append").format("delta")\
                        .option("path", "abfss://silver@spotifyazurestoragesink.dfs.core.windows.net/DimArtist/Data")\
                        .saveAsTable("spotify.silver.DimArtist")



**Dim Track**

In [0]:
df_track = spark.read.format("parquet").load("abfss://bronze@spotifyazurestoragesink.dfs.core.windows.net/DimTrack")
df_track.createOrReplaceTempView("df_track")

In [0]:
%sql 
select count(track_id) 
from df_track
group by track_id
having count(track_id) > 1

In [0]:
display(df_track)

In [0]:
df_track_transformed = df_track.withColumn("album_name", upper(col("album_name")))
df_track_transformed = df_track_transformed.dropDuplicates(["track_id"])
df_track_transformed = df_track_transformed.withColumn("Duration_Flag", when(col("duration_sec") < 150, "Low")\
                                                                    .when(col("duration_sec") < 300, "Medium")\
                                                                    .otherwise("High"))
df_track_transformed = df_track_transformed.withColumn("track_name", regexp_replace(col("track_name"), '-', ' '))
display(df_track_transformed)

In [0]:
df_track_transformed.write.mode("append").format("delta")\
                        .option("path", "abfss://silver@spotifyazurestoragesink.dfs.core.windows.net/DimTrack/Data")\
                        .saveAsTable("spotify.silver.dimtrack")

**Dim Date**

In [0]:
df_date = spark.read.format("parquet").load("abfss://bronze@spotifyazurestoragesink.dfs.core.windows.net/DimDate")


In [0]:
df_date_transformed = df_date.drop(col("day"),col("month"))
display(df_date_transformed)


In [0]:
df_date_transformed.write.mode("append").format("delta")\
                        .option("path", "abfss://silver@spotifyazurestoragesink.dfs.core.windows.net/DimDate/Data")\
                        .saveAsTable("spotify.silver.dimdate")

**Fact Stream**

In [0]:
df_fact = spark.read.format("parquet").load("abfss://bronze@spotifyazurestoragesink.dfs.core.windows.net/FactStream")


In [0]:
df_fact.write.mode("append").format("delta")\
                        .option("path", "abfss://silver@spotifyazurestoragesink.dfs.core.windows.net/FactStream/Data")\
                        .saveAsTable("spotify.silver.factstream")